# Lesson 03 — 计时，但不要把程序卡住

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


桌面提醒器开始倒计时后，仍必须响应取消按钮。把 `delay(3000)` 放进 `tick()` 会让程序在等待期间失去响应。

## 最小知识

非阻塞计时保存截止时间，而不是停住 CPU。每次 `tick()` 只比较 `now_ms` 与 deadline；状态机在多次短调用之间继续工作。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 03-nonblocking-timer guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            bool p = b_.read_button() == Level::low;
            if (p && !last_) {
                on_ = !on_;
            }
            last_ = p;
            b_.output.led = on_;
            b_.output.state = on_ ? "running" : "idle";
        }

    private:
        Device& b_;
        bool last_ = false, on_ = false;
    };
}


## 2. 修改一次

把 `3000` 改成 `1000`，观察剩余时间和完成状态。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 03-nonblocking-timer modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            bool p = b_.read_button() == Level::low;
            if (p && !last_) {
                on_ = true;
                end_ = b_.input.now_ms + 1000;
            }
            last_ = p;
            if (on_ && b_.input.now_ms >= end_) {
                on_ = false;
                b_.output.state = "done";
            }
            b_.output.led = on_;
        }

    private:
        Device& b_;
        bool last_ = false, on_ = false;
        unsigned end_ = 0;
    };
}


## 3. 故障实验（Debug）

下面代码一旦开始就永远保持 running，既没有超时，也无法被第二次按下取消。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 03-nonblocking-timer broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            bool p = b_.read_button() == Level::low;
            if (p && !last_ && !on_) {
                on_ = true;
            }
            last_ = p;
            b_.output.led = on_;
            b_.output.state = on_ ? "running" : "idle";
        }

    private:
        Device& b_;
        bool last_ = false, on_ = false;
    };
}


## 4. 你来完成（Build）

实现 3 秒倒计时：新按下开始，运行时的新按下取消；每次 tick 更新剩余毫秒，不等待、不循环。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/03-nonblocking-timer/hints.md)。


In [ ]:
%%academy_lab 03-nonblocking-timer exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() { // TODO: 保存截止时间；不要在这里等待
            b_.output.state = "idle";
        }

    private:
        Device& b_;
    };
}


## 5. 开放挑战

增加“暂停/继续”而不让暂停时间算进倒计时。先画状态图，再决定要保存哪些时间戳。

## 你刚刚真正学到了什么？

Timer 不是 sleep。你把“现在是否该发生某事”放进状态和时间比较，使主循环可以继续处理其他输入。

## 硬件迁移

STEP 2 用 `esp_timer_get_time()`、Arduino `millis()` 或 FreeRTOS software timer 提供单调时间。仍要避免在任务里长时间阻塞。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/03-nonblocking-timer-solution.ipynb)。
